In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv")

df[df["genres"].isna()].shape

(160910, 7)

In [5]:
df[df["genres"].isna()]["book_id"].nunique()

585

In [6]:
import pandas as pd
import ast

print("\nDataset shape:")
print(df.shape)


# -------------------------------------------------
# Genre parser (same logic as your SVD script)
# -------------------------------------------------
def parse_genres(s):

    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip().strip('"').strip("'") for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip().strip('"').strip("'") for t in s.split(sep) if t.strip()]

    return [s.strip().strip('"').strip("'")]


# -------------------------------------------------
# Collect statistics
# -------------------------------------------------

all_genres = set()
genre_book_map = {}
books_with_no_genre = set()

for _, row in df.iterrows():

    book = row["book_id"]
    parsed = parse_genres(row["genres"])

    if len(parsed) == 0:
        books_with_no_genre.add(book)
        continue

    for g in parsed:

        all_genres.add(g)

        if g not in genre_book_map:
            genre_book_map[g] = set()

        genre_book_map[g].add(book)


# -------------------------------------------------
# Unique genres
# -------------------------------------------------

print("\nUnique genres:")
for g in sorted(all_genres):
    print(g)

print("\nNumber of unique genres:", len(all_genres))


# -------------------------------------------------
# Unique books per genre
# -------------------------------------------------

print("\nUnique books per genre:")

genre_counts = []

for g in sorted(genre_book_map.keys()):
    count = len(genre_book_map[g])
    genre_counts.append((g, count))

genre_df = pd.DataFrame(genre_counts, columns=["genre", "unique_books"])
genre_df = genre_df.sort_values("unique_books", ascending=False)

print(genre_df)


# -------------------------------------------------
# Missing genres
# -------------------------------------------------

print("\nBooks with missing / unparsable genres:")
print(len(books_with_no_genre))


# -------------------------------------------------
# Dataset summary
# -------------------------------------------------

print("\nTotal unique books in dataset:")
print(df["book_id"].nunique())


Dataset shape:
(5976479, 7)

Unique genres:
Adult
Adventure
Children's
Classics
Drama
Fantasy
Historical
Horror
Mystery
Nonfiction
Romance
Science Fiction
Thriller

Number of unique genres: 13

Unique books per genre:
              genre  unique_books
4             Drama          3006
8           Mystery          2563
10          Romance          2131
5           Fantasy          2088
1         Adventure          1789
12         Thriller          1606
9        Nonfiction          1071
3          Classics           901
2        Children's           863
6        Historical           857
11  Science Fiction           855
7            Horror           769
0             Adult           331

Books with missing / unparsable genres:
585

Total unique books in dataset:
10000


In [8]:
import pandas as pd
import ast



def parse_genres(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# rows where genre parsing fails
invalid_rows = df[df["genres"].apply(lambda x: len(parse_genres(x)) == 0)]

print("Number of rows with invalid genres:", len(invalid_rows))

print("\nExample rows (full rows):\n")
print(invalid_rows.head(10))

Number of rows with invalid genres: 160910

Example rows (full rows):

      user_id  book_id  rating         decade original_title authors genres
1           2     4081       4           2000            NaN     NaN    NaN
433        32     1383       3           1970            NaN     NaN    NaN
438        32       75       3           1990            NaN     NaN    NaN
584        31       75       3           1990            NaN     NaN    NaN
1003       76       75       5           1990            NaN     NaN    NaN
1034       25     8555       4           1960            NaN     NaN    NaN
1297       78     4106       4  Ancient Books            NaN     NaN    NaN
1561       24     2757       3           1990            NaN     NaN    NaN
1619      113     3244       5           2000            NaN     NaN    NaN
1637      113     5629       4           1990            NaN     NaN    NaN


## clean dataset

In [9]:
import pandas as pd
import ast

INPUT_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv"
OUTPUT_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv"

df = pd.read_csv(INPUT_PATH)

print("Original dataset shape:", df.shape)


# -------------------------------------------------
# Genre parser (same logic as your SVD code)
# -------------------------------------------------
def parse_genres(s):

    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# -------------------------------------------------
# Identify books with no valid genre
# -------------------------------------------------

df["parsed_genres"] = df["genres"].apply(parse_genres)

invalid_books = df[df["parsed_genres"].apply(len) == 0]["book_id"].unique()

print("Books with no valid genre:", len(invalid_books))


# -------------------------------------------------
# Remove those books from the dataset
# -------------------------------------------------

df_clean = df[~df["book_id"].isin(invalid_books)].copy()

# drop helper column
df_clean.drop(columns=["parsed_genres"], inplace=True)

print("Clean dataset shape:", df_clean.shape)


# -------------------------------------------------
# Save cleaned dataset
# -------------------------------------------------

df_clean.to_csv(OUTPUT_PATH, index=False)

print("\nClean dataset saved as:", OUTPUT_PATH)

Original dataset shape: (5976479, 7)
Books with no valid genre: 585
Clean dataset shape: (5815569, 7)

Clean dataset saved as: /home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv


In [10]:
import pandas as pd
import ast

df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv")

def parse_genres(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# rows where genre parsing fails
invalid_rows = df[df["genres"].apply(lambda x: len(parse_genres(x)) == 0)]

print("Number of rows with invalid genres:", len(invalid_rows))

print("\nExample rows (full rows):\n")
print(invalid_rows.head(10))

Number of rows with invalid genres: 0

Example rows (full rows):

Empty DataFrame
Columns: [user_id, book_id, rating, decade, original_title, authors, genres]
Index: []


In [4]:
df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv")

df.head(10)

,user_id,book_id,rating,decade,original_title,authors,genres
0,1,258,5,2000,La sombra del viento,"Carlos Ruiz Zafón, Lucia Graves","Mystery, Historical"
1,2,260,5,1930,How to Win Friends and Influence People,Dale Carnegie,"Nonfiction, Drama"
2,2,9296,5,1970,Das Drama des begabten Kindes und die Suche na...,"Alice Miller, Ruth Ward","Horror, Mystery"
3,2,2318,3,1990,The Millionaire Next Door: The Surprising Secr...,"Thomas J. Stanley, William D. Danko","Nonfiction, Drama"
4,2,26,4,2000,The Da Vinci Code,Dan Brown,"Mystery, Thriller"
5,2,315,3,1990,Who Moved My Cheese?,"Spencer Johnson, Kenneth H. Blanchard","Nonfiction, Drama"
6,2,33,4,1990,Memoirs of a Geisha,Arthur Golden,"Romance, Historical"
7,2,301,5,Ancient Books,Heart of Darkness,Joseph Conrad,"Classics, Horror"
8,2,2686,5,2000,Blue Ocean Strategy: How to Create Uncontested...,"W. Chan Kim, Renée Mauborgne","Thriller, Mystery"
9,2,3753,5,2000,"Harry Potter Collection (Harry Potter, #1-6)",J.K. Rowling,"Mystery, Thriller"


In [9]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# =========================
# Paths
# =========================

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Load data
# =========================

df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv")

# =========================
# Basic stats
# =========================

num_users = df["user_id"].nunique()
num_books = df["book_id"].nunique()
num_ratings = len(df)

rating_distribution = df["rating"].value_counts().sort_index()

books_per_user = df.groupby("user_id")["book_id"].nunique()
min_books_per_user = books_per_user.min()
max_books_per_user = books_per_user.max()

# =========================
# Figure 1: Bar (Users, Books, Ratings)
# =========================

labels = ["Users", "Books", "Ratings"]
values = [num_users, num_books, num_ratings]

x = np.arange(len(labels)) * 1.5

fig, ax = plt.subplots(figsize=(8, 6))

bars = ax.bar(
    x,
    values,
    width=0.6,
    color=["#8ECae6", "#FFB703", "#B8DE6F"],
    edgecolor="black",
    linewidth=0.6
)

# STYLE
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_title("Dataset Size Overview", fontsize=14)
ax.set_ylabel("Count")

ax.set_xticks(x)
ax.set_xticklabels(labels)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        h,
        f"{int(h)}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_dataset_overview_bar.png", dpi=300)
plt.close()

# =========================
# Figure 2: Dot plot (Rating distribution)
# =========================

ratings = rating_distribution.index
counts = rating_distribution.values

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    ratings,
    counts,
    s=80,
    color="#8ECae6",
    edgecolor="black",
    linewidth=0.6
)

# STYLE
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_title("Rating Distribution (Dot Plot)", fontsize=14)
ax.set_xlabel("Rating Value")
ax.set_ylabel("Count")

for i, txt in enumerate(counts):
    ax.text(ratings[i], counts[i], str(txt),
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig2_rating_distribution_dot.png", dpi=300)
plt.close()

# =========================
# Figure 3: Min vs Max books per user (bar)
# =========================

labels = ["Min Books/User", "Max Books/User"]
values = [min_books_per_user, max_books_per_user]

x = np.arange(len(labels)) * 1.5

fig, ax = plt.subplots(figsize=(8, 6))

bars = ax.bar(
    x,
    values,
    width=0.6,
    color=["#F4A261", "#CDB4DB"],
    edgecolor="black",
    linewidth=0.6
)

# STYLE
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_title("User Activity Range", fontsize=14)
ax.set_ylabel("Number of Books Rated")

ax.set_xticks(x)
ax.set_xticklabels(labels)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        h,
        f"{int(h)}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "fig3_user_activity_bar.png", dpi=300)
plt.close()

# =========================
# Save summary table
# =========================

summary = pd.DataFrame({
    "metric": [
        "num_users",
        "num_books",
        "num_ratings",
        "min_books_per_user",
        "max_books_per_user"
    ],
    "value": [
        num_users,
        num_books,
        num_ratings,
        min_books_per_user,
        max_books_per_user
    ]
})

summary.to_csv(OUT_DIR / "dataset_summary.csv", index=False)

print("Saved files to:")
print(OUT_DIR)



Saved files to:
/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Patch
import numpy as np

# =========================
# Paths
# =========================

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Split genre pairs
# =========================

genres_split = df["genres"].astype(str).str.split(",", n=1, expand=True)
df["g1"] = genres_split[0].str.strip()
df["g2"] = genres_split[1].str.strip() if 1 in genres_split.columns else ""

books = df[["book_id", "g1", "g2"]].drop_duplicates(subset=["book_id"]).copy()

total_books = books["book_id"].nunique()

# =========================
# Counts
# =========================

first_counts = books["g1"].value_counts()
second_counts = books["g2"].value_counts()

all_genres = sorted(set(first_counts.index).union(set(second_counts.index)))

first_counts = first_counts.reindex(all_genres, fill_value=0)
second_counts = second_counts.reindex(all_genres, fill_value=0)
total_counts = first_counts + second_counts

grand_total_genre_appearances = total_counts.sum()
pct_all = (total_counts / grand_total_genre_appearances) * 100

pct_first_of_genre = (first_counts / total_counts * 100).fillna(0)
pct_second_of_genre = (second_counts / total_counts * 100).fillna(0)

# =========================
# Summary table
# =========================

summary = pd.DataFrame({
    "genre": all_genres,
    "first": first_counts.values,
    "second": second_counts.values,
    "total": total_counts.values,
    "percentage_of_all": pct_all.values,
    "percentage_of_that_genre_first": pct_first_of_genre.values,
    "percentage_of_that_genre_second": pct_second_of_genre.values,
}).sort_values("total", ascending=False)

summary.to_csv(OUT_DIR / "genre_summary_table.csv", index=False)

# =========================
# Soft color palette
# =========================

colors = ["#8ECae6", "#FFB703", "#B8DE6F", "#F4A261", "#CDB4DB"]
bar_colors = [colors[i % len(colors)] for i in range(len(summary))]

# =========================
# Figure 1: Total books per genre
# =========================

x = np.arange(len(summary)) * 1.4
width = 0.7

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.bar(
    x,
    summary["total"],
    width=width,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.6
)

# STYLE
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_title("Total Number of Unique Books per Genre", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Unique Books")

ax.set_xticks(x)
ax.set_xticklabels(summary["genre"], rotation=90)

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        h + max(summary["total"]) * 0.01,
        f"{int(h)}",
        ha="center",
        va="bottom",
        fontsize=8
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "fig1_total_books_per_genre_bar.png", dpi=300)
plt.close()

# =========================
# Figure 2: Pie (unchanged)
# =========================

fig, ax = plt.subplots(figsize=(10, 10))
ax.pie(
    summary["total"],
    labels=summary["genre"],
    autopct="%1.1f%%",
    startangle=90,
    colors=bar_colors,
    wedgeprops={"edgecolor": "white", "linewidth": 1}
)

ax.set_title("Percentage of All Genre Appearances", fontsize=14)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig2_genre_percentage_pie.png", dpi=300)
plt.close()

# =========================
# Figure 3: Grouped bar
# =========================

plot_df = summary.copy()

x = np.arange(len(plot_df)) * 1.4
width = 0.35

fig, ax = plt.subplots(figsize=(13, 7))

bars1 = ax.bar(
    x - width/2,
    plot_df["first"],
    width=width,
    color="#8ECae6",
    edgecolor="black",
    linewidth=0.6,
    label="First genre"
)

bars2 = ax.bar(
    x + width/2,
    plot_df["second"],
    width=width,
    color="#F4A261",
    edgecolor="black",
    linewidth=0.6,
    label="Second genre"
)

# STYLE
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_title("Genre Counts by Position", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Unique Books")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["genre"], rotation=90)

ax.legend()

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h, f"{int(h)}",
            ha="center", va="bottom", fontsize=8)

for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h, f"{int(h)}",
            ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig3_first_vs_second_grouped_bar.png", dpi=300)
plt.close()

# =========================
# Figure 4: Stacked bar
# =========================

fig, ax = plt.subplots(figsize=(13, 7))

bars_first = ax.bar(
    x,
    plot_df["first"],
    color="#8ECae6",
    edgecolor="black",
    linewidth=0.6,
    label="First genre"
)

bars_second = ax.bar(
    x,
    plot_df["second"],
    bottom=plot_df["first"],
    color="#F4A261",
    edgecolor="black",
    linewidth=0.6,
    label="Second genre"
)

# STYLE
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_title("First and Second Genre Distribution per Genre", fontsize=14)
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Unique Books")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["genre"], rotation=90)

ax.legend()

for i, total in enumerate(plot_df["total"]):
    ax.text(x[i], total, f"{int(total)}",
            ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig4_first_second_stacked_bar.png", dpi=300)
plt.close()

# =========================
# Save requested CSV
# =========================

requested_table = summary[[
    "genre", "first", "second", "total",
    "percentage_of_all", "percentage_of_that_genre_first"
]].copy()

requested_table = requested_table.rename(columns={
    "percentage_of_that_genre_first": "percentage_of_that_genre"
})

requested_table.to_csv(OUT_DIR / "genre_requested_table.csv", index=False)

print("Saved files to:")
print(OUT_DIR)

Saved files to:
/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures


In [12]:

import pandas as pd
import os

# Load dataset
df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/data/genre_clean.csv")

# 1. Number of unique users
num_users = df['user_id'].nunique()

# 2. Number of unique books
num_books = df['book_id'].nunique()

# 3. Number of ratings
num_ratings = len(df)

# 4. Rating style (distribution of ratings)
rating_distribution = df['rating'].value_counts().sort_index()

# 5. Lowest number of books rated by a single user
books_per_user = df.groupby('user_id')['book_id'].nunique()
min_books_per_user = books_per_user.min()

# 6. Highest number of books rated by a single user
max_books_per_user = books_per_user.max()

# Print results
print("Number of unique users:", num_users)
print("Number of unique books:", num_books)
print("Total number of ratings:", num_ratings)
print("\nRating distribution:\n", rating_distribution)
print("\nMinimum books rated by a user:", min_books_per_user)
print("Maximum books rated by a user:", max_books_per_user)

# Save results
output_dir = "/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures"
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, "dataset_analysis.txt"), "w") as f:
    f.write(f"Number of unique users: {num_users}\n")
    f.write(f"Number of unique books: {num_books}\n")
    f.write(f"Total number of ratings: {num_ratings}\n\n")
    f.write("Rating distribution:\n")
    f.write(rating_distribution.to_string())
    f.write("\n\n")
    f.write(f"Minimum books rated by a user: {min_books_per_user}\n")
    f.write(f"Maximum books rated by a user: {max_books_per_user}\n")


Number of unique users: 53424
Number of unique books: 9415
Total number of ratings: 5815569

Rating distribution:
 rating
1     120964
2     350205
3    1336839
4    2083651
5    1923910
Name: count, dtype: int64

Minimum books rated by a user: 18
Maximum books rated by a user: 198


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# =========================
# Paths
# =========================

OUT_DIR = Path("/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Data
# =========================

ratings = [1, 2, 3, 4, 5]
counts = [120964, 350205, 1336839, 2083651, 1923910]

# Summary info
num_users = 53424
num_books = 9415
num_ratings = 5815569

# =========================
# Figure: Bar plot
# =========================

fig, ax = plt.subplots(figsize=(10, 6))

# Gray background
fig.patch.set_facecolor("#f2f2f2")
ax.set_facecolor("#f2f2f2")

x = np.arange(len(ratings))

bars = ax.bar(
    x,
    counts,
    width=0.6,
    color="#6c757d",
    edgecolor="black",
    linewidth=0.7
)

# =========================
# STYLE
# =========================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_linewidth(1.2)
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_position(("outward", 5))

ax.yaxis.grid(True, linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

ax.set_title("Rating Distribution", fontsize=14, color="#333333")
ax.set_xlabel("Rating", color="#333333")
ax.set_ylabel("Number of Ratings", color="#333333")

ax.set_xticks(x)
ax.set_xticklabels(ratings)

ax.tick_params(axis='x', colors="#333333")
ax.tick_params(axis='y', colors="#333333")

# =========================
# Bar labels
# =========================

for bar in bars:
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        h + max(counts) * 0.01,
        f"{int(h):,}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#333333"
    )

# =========================
# Info box (top right)
# =========================

info_text = (
    f"Users: {num_users:,}\n"
    f"Books: {num_books:,}\n"
    f"Ratings: {num_ratings:,}"
)

ax.text(
    0.98, 0.98,
    info_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=10,
    color="#333333",
    bbox=dict(
        facecolor="white",
        edgecolor="gray",
        boxstyle="round,pad=0.3",
        alpha=0.95
    )
)

# =========================
# Save
# =========================

plt.tight_layout()
plt.savefig(OUT_DIR / "fig_rating_distribution_bar.png", dpi=300, facecolor=fig.get_facecolor())
plt.close()

print("Saved to:")
print(OUT_DIR)

Saved to:
/home/moshtasa/Research/phd-svd-recsys/Book/0311_similar_pr_details/Data/domain_analysis/figures
